# WRDS 历史数据下载、处理与更新

此 notebook 调用项目已有的 `scripts.data_pipeline` 模块，下载 **2022-01-03 起**的 NYSE/Nasdaq 日频数据及 TAQ 日内汇总，并保存到本地 `Data/`。

- **首次使用**：依次执行「1 环境」「2 参数与登录方法」「3 下载」「4 处理」。
- **后续更新**：执行「1 环境」「2 参数与登录方法」「5 更新」，自动补洞、刷新近期数据并处理。
- 「6 审计与预览」可单独检查本地数据，无需登录。

账号和密码只在运行时输入。密码隐藏；需要时请在自己的设备上完成 Duo 批准。不要在 cell 中填写真实账号或密码。此文件不包含运行输出，`Data/` 已被 Git 忽略。

请选择安装了项目依赖的 Python 内核，例如 `.venv-forecast`。首次准备环境可在项目根目录的终端执行：

```bash
.venv-forecast/bin/python -m pip install -r scripts/download/requirements-wrds.txt
```

下载为真实 WRDS 请求；多年全市场下载可能耗时较长。日内数据为服务器端汇总的检查点和竞价量，并非完整逐笔文件。

## 1. 环境与项目路径

自动从当前工作目录向上寻找项目。可以从仓库根目录或 `scripts/download/` 启动。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "scripts/data_pipeline/pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("未找到项目根目录；请从本仓库中启动 notebook。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import pandas as pd
    from IPython.display import display
    from scripts.data_pipeline import update_data, process_data, audit_data
except ImportError as exc:
    raise ImportError(
        "请在当前 notebook 内核中安装 scripts/download/requirements-wrds.txt 的依赖。"
    ) from exc

print("项目目录：", PROJECT_ROOT)
print("当前内核：", sys.executable)

## 2. 参数与登录方法

默认保留从 2022 年开始的历史，截止日期 `None` 表示更新到各来源最新可用的完整交易日。`MARKETS` 可改为 `("NYSE",)`。首次小范围验证可以临时将 `END_DATE` 改成 `"2022-01-14"`；验证后恢复 `None`。

认证回调适用于 Jupyter，无需终端的 `isatty` 支持。每次下载或更新会询问账号和隐藏密码；本 cell 仅定义方法，不进行登录。

In [ ]:
from getpass import getpass

START_DATE = "2022-01-03"
END_DATE = None
MARKETS = ("NYSE", "NASDAQ")
DATA_DIR = PROJECT_ROOT / "Data"
REFRESH_SESSIONS = 5


def notebook_credentials():
    print("请输入自己的 WRDS 账号，并在需要时完成 Duo 批准。")
    username = input("WRDS username: ").strip()
    password = getpass("WRDS password（隐藏输入）: ")
    if not username or not password:
        raise ValueError("账号和密码不能为空。")
    return {"user": username, "password": password}


def show_update_report(report):
    # 只显示不含认证信息的摘要，不输出凭证或数千条分区记录。
    display({
        "status": report.get("status"),
        "complete": report.get("complete"),
        "stage": report.get("stage"),
        "common_end": report.get("common_end"),
        "error": report.get("error"),
    })
    if report.get("failure_detail"):
        display(report["failure_detail"])
    if report.get("sources"):
        display(pd.DataFrame.from_dict(report["sources"], orient="index"))
    if report.get("chunks"):
        chunks = pd.DataFrame(report["chunks"])
        display(chunks.groupby(["kind", "status"]).size().rename("partitions").to_frame())
        failed = chunks.loc[chunks["status"].eq("failed")]
        if not failed.empty:
            display(failed)
    if not report.get("complete"):
        print("本次请求未全部完成；请检查 pending 日期、失败分区或质量未决项。")
    print("详细报告：", DATA_DIR / "update_report.json")


print({"start": START_DATE, "end": END_DATE, "markets": MARKETS, "data_dir": str(DATA_DIR)})

## 3. 首次下载：仅获取并验证 raw 数据

执行后才开始登录和下载。已通过校验的历史分区会复用；缺失或损坏分区会补取。首次下载关闭近期强制刷新，处理工作留给下一 cell。

失败后可以重跑本 cell 恢复。`complete=False` 也可能表示请求截止日尚未全部发布，请结合报告中的逐来源覆盖和 `pending_sessions` 判断。

In [ ]:
download_report = update_data(
    start=START_DATE,
    end=END_DATE,
    markets=MARKETS,
    data_dir=DATA_DIR,
    refresh_sessions=0,
    download_only=True,
    credentials_provider=notebook_credentials,
)
show_update_report(download_report)

## 4. 离线处理：生成标准化数据和质量筛选结果

本 cell **不联网**，读取最近一次下载计划及已验证的 raw 分区；修改上方日期变量不会改变已有下载计划。先通过审计，再发布处理结果，失败保留上一套有效版本。

质量规则：日频总成交量和竞价量任一 ≤0、NA 或无穷即为异常；过去 10 个交易日（不含当日）至少 6 天有效，当日也必须有效；有效上市/市场区间全历史异常率 >5% 时整只剔除。此筛选使用全历史信息，不能直接作为严格历史时点回测的股票池。

In [ ]:
raw_audit = audit_data(data_dir=DATA_DIR)
display(raw_audit)
if not raw_audit["complete"]:
    raise RuntimeError("本地 raw 分区未通过审计；请重跑下载 cell 修复后再处理。")

processing_report = process_data(data_dir=DATA_DIR)
display({
    "quality_status": processing_report["quality_status"],
    "unresolved_securities": processing_report["unresolved_securities"],
    "reused": processing_report.get("reused", False),
    "selection_policy": processing_report["selection_policy"],
})
display(pd.DataFrame([
    {"file": name, "rows": info["rows"]}
    for name, info in processing_report["files"].items()
]))
if processing_report["quality_status"] != "complete":
    print("存在质量未决证券，请检查 us_wrds_quality/security_quality.csv。")

## 5. 后续更新：补洞、追加、刷新并处理

后续使用时执行本 cell 即可，不需要同时重跑首次下载和离线处理 cell。仍从固定的 2022 年起点核对覆盖，自动跳过有效历史分区，刷新最近 `REFRESH_SESSIONS` 个已发布交易日，并发布新处理结果。

`REFRESH_FROM = None` 表示只按近期窗口刷新；若要重查更早的历史修订，可设置为相应日期（例如 `"2022-01-03"`，将触发大范围刷新）。更新期间不要在其他进程中同时写入相同的数据目录。

In [ ]:
REFRESH_FROM = None

update_report = update_data(
    start=START_DATE,
    end=END_DATE,
    markets=MARKETS,
    data_dir=DATA_DIR,
    refresh_sessions=REFRESH_SESSIONS,
    refresh_start=REFRESH_FROM,
    download_only=False,
    credentials_provider=notebook_credentials,
)
show_update_report(update_report)
if update_report.get("processing"):
    display({
        "quality_status": update_report["processing"]["quality_status"],
        "unresolved_securities": update_report["processing"]["unresolved_securities"],
        "reused": update_report["processing"].get("reused", False),
    })

## 6. 本地审计与结果预览

审计检查下载分区；处理清单另外列出产物和质量状态。预览只读取少量行。先固定当前版本的实际路径，避免读取多个文件时跨越不同版本。

In [ ]:
import json

display(audit_data(data_dir=DATA_DIR))
current = DATA_DIR / "processed/wrds_current"
if not current.exists():
    print("尚无新版处理产物，请先完成下载并运行处理 cell。")
else:
    version_dir = current.resolve(strict=True)
    manifest = json.loads((version_dir / "manifest.json").read_text())
    print("当前数据版本：", version_dir)
    display({
        "processing_version": manifest["version"],
        "quality_status": manifest["quality_status"],
        "deferred_securities": manifest["deferred_securities"],
        "deferred_reasons": manifest["deferred_reasons"],
        "unverified_events": manifest["unverified_events"],
        "selection_policy": manifest["selection_policy"],
    })
    # Securities / status alignment and deferred (unresolved) securities kept with their reasons.
    display(pd.read_csv(version_dir / "report/security_status_summary.csv"))
    for relative_path in (
        "us_wrds_daily/daily_clean.csv.gz",
        "us_wrds_taq/checkpoints_research.csv.gz",
        "us_wrds_quality/security_quality.csv",
        "us_wrds_quality/research_sample.csv.gz",
    ):
        print(relative_path)
        display(pd.read_csv(
            version_dir / relative_path,
            nrows=5,
            keep_default_na=False,
            na_values=[""],
            dtype={"security_id": str, "gvkey": str, "iid": str},
        ))
    # Training filter: research_sample already holds only research_included_expost | research_included_pit rows.
    # Choose ONE flag for a given experiment; pit_eligible alone is the strict PIT definition, not the training filter.

更多字段定义、认证说明与失败恢复方法见 [WRDS 数据流程](../../docs/WRDS_DATA_PIPELINE.md)。本 notebook 不写入账号或密码；共享前清除运行输出，保留数据文件在已忽略的 `Data/` 中。